# Análisis comercial con SQL

## Objetivo

Analizar las transacciones de la tienda online con SQL para estudiar la evolución de las ventas, los productos, los países y los clientes.

## 1. Carga y preparación de los datos

Se carga el archivo parquet limpio y se prepara una copia para MySQL. El archivo original no se modifica.

In [2]:
import pandas as pd

df= pd.read_csv(
    "../data/processed/online_retail_clean.csv")

print("Número de filas:", len(df))
print("Número de columnas:", len(df.columns))
print("\nColumnas y su tipo:")
df.info()

display(df.head())

Número de filas: 536641
Número de columnas: 14

Columnas y su tipo:
<class 'pandas.DataFrame'>
RangeIndex: 536641 entries, 0 to 536640
Data columns (total 14 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Unnamed: 0          536641 non-null  int64  
 1   InvoiceNo           536641 non-null  str    
 2   StockCode           536641 non-null  str    
 3   Description         535187 non-null  str    
 4   Quantity            536641 non-null  int64  
 5   InvoiceDate         536641 non-null  str    
 6   UnitPrice           536641 non-null  float64
 7   CustomerID          401604 non-null  float64
 8   Country             536641 non-null  str    
 9   IsCancelled         536641 non-null  bool   
 10  IsNegativeQuantity  536641 non-null  bool   
 11  IsZeroPrice         536641 non-null  bool   
 12  IsNegativePrice     536641 non-null  bool   
 13  LineTotal           536641 non-null  float64
dtypes: bool(4), float64(3), int

,Unnamed: 0,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancelled,IsNegativeQuantity,IsZeroPrice,IsNegativePrice,LineTotal
0,0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,False,False,False,15.30
1,1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,False,20.34
2,2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,False,False,False,22.00
3,3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,False,20.34
4,4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,False,20.34


### Preparación del formato para MySQL

Los indicadores booleanos se convierten a 0 y 1 para su almacenamiento en MySQL. Los valores ausentes se conservarán como nulos.

In [3]:
columnas_bool = [
    "IsCancelled",
    "IsNegativeQuantity",
    "IsZeroPrice",
    "IsNegativePrice"
]

df_mysql = df.copy()

for col in columnas_bool:
    df_mysql[col] = df_mysql[col].astype("int8")

print("Filas preparadas:", len(df_mysql))
print("Columnas preparadas:", len(df_mysql.columns))


Filas preparadas: 536641
Columnas preparadas: 14


## 2. Importación y validación en MySQL

Los datos se insertan desde df_mysql mediante mysql-connector-python. Los valores ausentes se convierten en NULL y las fechas e importes se adaptan a los tipos de MySQL.

Importante: ejecutar la inserción una sola vez y solo si la tabla está vacía.

### Método de importación: Python frente a Table Data Import Wizard

Se ha elegido `mysql-connector-python` en lugar de **Table Data Import Wizard** de MySQL Workbench por dos motivos principales:

- **Control de los datos:** durante las pruebas, el asistente presentó problemas al interpretar los valores nulos de `CustomerID`. Con Python podemos convertir explícitamente los valores ausentes a `NULL` de MySQL y adaptar las fechas, los importes y los indicadores `0/1` a los tipos de la tabla.
- **Velocidad de carga:** el conector permite insertar los 536.641 registros por lotes (`executemany`), reduciendo las operaciones individuales y el trabajo manual de la interfaz gráfica. Esto hace que la carga sea más eficiente para un conjunto de este tamaño.

In [4]:
import mysql.connector
from decimal import Decimal

conn = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=input("Introduce la contraseña de MySQL: "),
    database="retail_sales"
)

cursor = conn.cursor()
print("Conexión a MySQL establecida correctamente.")


Conexión a MySQL establecida correctamente.


In [5]:
# Evitar una segunda importación accidental
cursor.execute("SELECT COUNT(*) FROM online_retail")
existing_rows = cursor.fetchone()[0]
if existing_rows > 0:
    raise RuntimeError(
        f"La tabla ya contiene {existing_rows:,} filas. "
        "No se insertarán datos para evitar duplicados."
    )

columnas = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country",
    "IsCancelled",
    "IsNegativeQuantity",
    "IsZeroPrice",
    "IsNegativePrice",
    "LineTotal"
]

# Preparar la sentencia SQL
placeholders = ", ".join(["%s"] * len(columnas))
nombre_columnas = ", ".join(f"`{col}`" for col in columnas)

sql = f"""
INSERT INTO online_retail ({nombre_columnas})
VALUES ({placeholders})
"""

# Convertir los valores ausentes a None y preparar los registros
df_insert = df_mysql[columnas].astype(object).where(
    df_mysql[columnas].notna(), None
)

# Convertir tipos para MySQL
df_insert["InvoiceDate"] = pd.to_datetime(
    df_insert["InvoiceDate"]
).dt.to_pydatetime()

for col in ["UnitPrice", "LineTotal"]:
    df_insert[col] = df_insert[col].map(
        lambda x: None if x is None else Decimal(str(x)).quantize(
            Decimal("0.01")
        )
    )

filas = list(df_insert.itertuples(index=False, name=None))

batch_size = 5000
total = 0

try:
    for start in range(0, len(filas), batch_size):
        batch = filas[start:start + batch_size]

        cursor.executemany(sql, batch)
        conn.commit()

        total += len(batch)
        if total % 50000 == 0 or total == len(filas):
            print(f"Insertadas {total:,} de {len(filas):,} filas")

    print("\nImportación completada.")
    print("Total de filas insertadas:", total)

except Exception:
    conn.rollback()
    raise


Insertadas 50,000 de 536,641 filas
Insertadas 100,000 de 536,641 filas
Insertadas 150,000 de 536,641 filas
Insertadas 200,000 de 536,641 filas
Insertadas 250,000 de 536,641 filas
Insertadas 300,000 de 536,641 filas
Insertadas 350,000 de 536,641 filas
Insertadas 400,000 de 536,641 filas
Insertadas 450,000 de 536,641 filas
Insertadas 500,000 de 536,641 filas
Insertadas 536,641 de 536,641 filas

Importación completada.
Total de filas insertadas: 536641


In [6]:
cursor.execute("""
    SELECT
        COUNT(*) AS total_filas,
        COUNT(CustomerID) AS clientes_identificados,
        SUM(CustomerID IS NULL) AS clientes_sin_id,
        SUM(Description IS NULL) AS descripciones_nulas
    FROM online_retail
""")

resultado = cursor.fetchone()

print("Total de filas:", resultado[0])
print("Filas con CustomerID:", resultado[1])
print("Filas sin CustomerID:", resultado[2])
print("Descripciones nulas:", resultado[3])

Total de filas: 536641
Filas con CustomerID: 401604
Filas sin CustomerID: 135037
Descripciones nulas: 1454


In [7]:
cursor.close()
conn.close()